# Earnings Predictor on Colab

Runs the Flask app from GitHub inside Colab.

**Before running**, add these in the 🔑 **Secrets** panel (left sidebar) and turn on notebook access for each:

| Name | Value |
|---|---|
| `SEC_USER_AGENT` | `Your Name your.email@example.com` |
| `SUPABASE_URL` | your Supabase project URL |
| `SUPABASE_SERVICE_KEY` | the service_role key |
| `GITHUB_TOKEN` | *(only if the repo is private)* a GitHub token with repo read access |

Then **Runtime → Run all**. The last cell prints a link to open the app.

## 1. Clone the repo

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name, default=None):
    try:
        return userdata.get(name)
    except Exception:
        return default

REPO = "github.com/Lukeharris33/earnings_predictor.git"
token = secret("GITHUB_TOKEN")
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

%cd /content
if os.path.isdir("earnings_predictor"):
    !git -C earnings_predictor pull -q
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", url, "earnings_predictor"], check=True)
%cd /content/earnings_predictor
!git log --oneline -1

## 2. Install dependencies
Colab already ships TensorFlow, so `tensorflow-cpu` and `gunicorn` are skipped. If Colab asks you to restart the runtime afterwards, do it and run all again.

In [ ]:
!grep -vE '^(tensorflow-cpu|gunicorn)' requirements.txt > /tmp/reqs.txt
!pip install -q -r /tmp/reqs.txt

## 3. Configure environment

In [ ]:
for name in ["SEC_USER_AGENT", "SUPABASE_URL", "SUPABASE_SERVICE_KEY", "SUPABASE_MODEL_BUCKET"]:
    val = secret(name)
    if val:
        os.environ[name] = val
    else:
        print(f"Warning: secret {name} not set")

os.environ.setdefault("SUPABASE_MODEL_BUCKET", "models")
os.environ["SECRET_KEY"] = os.urandom(16).hex()
os.environ["FLASK_HOST"] = "0.0.0.0"
os.environ["FLASK_PORT"] = "5000"
os.environ["FLASK_DEBUG"] = "false"
os.environ["CACHE_DIR"] = "/content/earnings_predictor/data/cache"

## 4. Start the app
Runs Flask in the background. Re-run this cell to restart it.

In [ ]:
!pkill -f "python app.py" || true
!nohup python app.py > /content/app.log 2>&1 &

import time, urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/", timeout=2)
        print("App is up")
        break
    except Exception:
        time.sleep(2)
else:
    print("App did not start; see the log below")
!tail -n 20 /content/app.log

## 5. Open it
Click the link below (opens in a new tab while this notebook stays connected).

In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

## Logs
Run any time to see what the app is doing.

In [ ]:
!tail -n 50 /content/app.log